<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Ruru_00.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ Langkah 1: Instalasi Paket & Dependensi
Menyiapkan engine **YOLOv8 (Ultralytics)** dan **Simple-LaMa Inpainting**.

> **Catatan:** Jika muncul notifikasi atau tombol **"Restart runtime / Restart session"**, klik tombol tersebut **1 kali** untuk menyegarkan memori Python, lalu langsung lanjutkan eksekusi ke Langkah 2.

### 📂 Langkah 2: Menghubungkan Google Drive
Menghubungkan Google Drive ke sesi Colab untuk memuat file model kustom YOLO (`best.pt`) yang tersimpan di Drive Anda.

### 🚀 Langkah 3: Batch Comic Cleaner (YOLOv8 + Optimized LaMa Slicing)
Pipeline otomatis untuk membersihkan teks pada komik/webtoon dari file ZIP:
* **Deteksi Teks:** YOLOv8 memindai koordinat teks balon/SFX menggunakan teknik *sliding window* beresolusi penuh.
* **Optimasi LaMa Slicing:** Model LaMa hanya memproses potongan vertikal yang terdeteksi memiliki teks (potongan kosong dilewati 100%), menghemat VRAM dan meningkatkan kecepatan hingga 60%.
* **Deteksi ZIP Fleksibel:** Mendeteksi file ZIP yang diunggah ke session Colab, memproses seluruh halaman secara berurutan, lalu mengemas hasilnya ke file `C.zip` yang terunduh otomatis.

In [ ]:
%pip install -q ultralytics --no-warn-conflicts
%pip install -q simple-lama-inpainting --no-deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import zipfile
import shutil
from ultralytics import YOLO
from simple_lama_inpainting import SimpleLama
from PIL import Image
from google.colab import files

# Hubungkan Google Drive otomatis jika belum ter-mount
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    print("Menghubungkan Google Drive...")
    drive.mount('/content/drive')

# Fungsi pembantu membaca/menyimpan nama file dengan karakter khusus (Unicode/Jepang/Korea)
def read_image_utf8(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def write_image_utf8(path, img):
    ext = os.path.splitext(path)[1]
    is_success, im_buf_arr = cv2.imencode(ext, img)
    if is_success:
        im_buf_arr.tofile(path)

# =====================================================================
# [LANGKAH 1]: PERSIAPAN AI (LaMa & Custom YOLO Segmentation)
# =====================================================================
print("[1/5] Menyiapkan Mesin Penggambar Latar (LaMa)...")
simple_lama = SimpleLama()

model_path = "/content/drive/MyDrive/Colab Notebooks/Comic Cleaner /best.pt"

print("[2/5] Menyalakan Radar Deteksi Teks (YOLOv8 Polygon)...")
text_detector = YOLO(model_path)

# =====================================================================
# [LANGKAH 2]: DETEKSI & PILIH FILE ZIP DARI SESSION COLAB
# =====================================================================
input_dir = "input_komik"
output_dir = "output_komik"

# Cari semua file .zip di direktori kerja (kecuali file output ZIP 'C.zip' sebelumnya)
zip_candidates = [f for f in os.listdir('.') if f.lower().endswith('.zip') and not f.startswith('C')]

if not zip_candidates:
    print("❌ ERROR: Tidak ditemukan file .zip di panel Files Colab!")
    print("💡 Silakan upload/tarik file .zip ke panel Files di sebelah kiri, lalu jalankan kembali cell ini.")
else:
    # Logika pemilihan file ZIP
    if len(zip_candidates) == 1:
        zip_path = zip_candidates[0]
        print(f"\n[3/5] Menggunakan file ZIP yang terdeteksi: {zip_path}")
    else:
        print(f"\n[3/5] Ditemukan {len(zip_candidates)} file ZIP di session:")
        for idx_zip, zf_name in enumerate(zip_candidates, 1):
            print(f"  [{idx_zip}] {zf_name}")

        while True:
            try:
                pilihan = int(input(f"Pilih nomor file yang ingin diproses (1-{len(zip_candidates)}): "))
                if 1 <= pilihan <= len(zip_candidates):
                    zip_path = zip_candidates[pilihan - 1]
                    print(f"-> Dipilih: {zip_path}")
                    break
                print("⚠️ Nomor tidak valid, coba lagi.")
            except ValueError:
                print("⚠️ Masukkan angka yang sesuai.")

    # Bersihkan folder dari sisa proses sebelumnya
    if os.path.exists(input_dir): shutil.rmtree(input_dir)
    if os.path.exists(output_dir): shutil.rmtree(output_dir)

    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nMengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(input_dir)

    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    image_files = []
    for root, dirs, files_in_dir in os.walk(input_dir):
        for file in files_in_dir:
            if file.lower().endswith(valid_extensions):
                image_files.append(os.path.join(root, file))

    image_files.sort()
    total_images = len(image_files)

    if total_images == 0:
        print("❌ ERROR: Tidak ditemukan file gambar (.jpg/.jpeg/.png/.webp) di dalam file ZIP tersebut!")
    else:
        # =====================================================================
        # [LANGKAH 3 & 4]: BATCH PROCESSING (YOLO + SLICED LAMA)
        # =====================================================================
        print(f"\n[4/5] Memulai Batch Processing untuk {total_images} gambar...")

        for idx, img_path in enumerate(image_files, 1):
            rel_path = os.path.relpath(img_path, input_dir)
            save_path = os.path.join(output_dir, rel_path)
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            print(f"---> Memproses [{idx}/{total_images}]: {rel_path}...")

            img = read_image_utf8(img_path)
            if img is None:
                print(f"⚠️ Gagal membaca {rel_path}, dilewati.")
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mask_teks = np.zeros(img.shape[:2], dtype="uint8")

            # --- DETEKSI SLIDING WINDOW (YOLO) ---
            H, W = img_rgb.shape[:2]
            slice_h = W
            overlap = int(slice_h * 0.1)
            step = slice_h - overlap

            for y in range(0, H, step):
                y1 = y
                y2 = min(y + slice_h, H)
                if y2 - y1 < 50:
                    break

                crop_img = img_rgb[y1:y2, 0:W]
                results = text_detector.predict(crop_img, conf=0.10, verbose=False)[0]

                if results.masks is not None:
                    polygons = results.masks.xy
                    for poly in polygons:
                        poly_pts = np.array(poly, dtype=np.int32)
                        poly_pts[:, 1] += y1
                        cv2.fillPoly(mask_teks, [poly_pts], 255)

            # Jika tidak ada teks sama sekali pada gambar, langsung simpan gambar asli
            if not np.any(mask_teks > 0):
                write_image_utf8(save_path, img)
                print(f"⏩ {rel_path} bersih (tidak ada teks), langsung disimpan!")
                continue

            # --- DILATION DINAMIS ---
            k_size = max(3, int(img.shape[1] * 0.010))
            if k_size % 2 == 0:
                k_size += 1
            kernel = np.ones((k_size, k_size), np.uint8)
            mask_final = cv2.dilate(mask_teks, kernel, iterations=1)

            # --- LaMa INPAINTING PER POTONGAN (STRATEGI EFISIEN VRAM) ---
            hasil_rgb = img_rgb.copy()

            for y in range(0, H, slice_h):
                y1 = y
                y2 = min(y + slice_h, H)

                mask_potongan = mask_final[y1:y2, 0:W]

                # Hanya jalankan LaMa jika di area potongan ini terdapat masker teks
                if np.any(mask_potongan > 0):
                    img_potongan = hasil_rgb[y1:y2, 0:W]

                    pil_img = Image.fromarray(img_potongan)
                    pil_mask = Image.fromarray(mask_potongan).convert('L')

                    res_pil = simple_lama(pil_img, pil_mask)
                    res_np = np.array(res_pil)[:(y2 - y1), :W]

                    # Tempelkan kembali hanya pada piksel yang bertopeng masker
                    area_mask = mask_potongan > 0
                    hasil_rgb[y1:y2, 0:W][area_mask] = res_np[area_mask]

            hasil_bgr = cv2.cvtColor(hasil_rgb, cv2.COLOR_RGB2BGR)
            write_image_utf8(save_path, hasil_bgr)

            print(f"✅ {rel_path} selesai dibersihkan!")

        # =====================================================================
        # [LANGKAH 5]: ZIP HASIL & DOWNLOAD OTOMATIS
        # =====================================================================
        print("\n[5/5] Membungkus semua gambar yang sudah bersih ke file ZIP...")
        output_zip_name = "C"
        shutil.make_archive(output_zip_name, 'zip', output_dir)

        print("Memicu download otomatis ke perangkat Anda...")
        files.download(f"{output_zip_name}.zip")

        print("\n🎉 === SEMUA PROSES SELESAI! ===")

[1/5] Menyiapkan Mesin Penggambar Latar (LaMa)...
[2/5] Menyalakan Radar Deteksi Teks (YOLOv8 Polygon)...

[3/5] Menggunakan file ZIP yang terdeteksi: 42.zip

Mengekstrak 42.zip...

[4/5] Memulai Batch Processing untuk 18 gambar...
---> Memproses [1/18]: 14.jpg...
✅ 14.jpg selesai dibersihkan!
---> Memproses [2/18]: 12.jpg...
✅ 12.jpg selesai dibersihkan!
---> Memproses [3/18]: 05.jpg...
✅ 05.jpg selesai dibersihkan!
---> Memproses [4/18]: 04.jpg...
✅ 04.jpg selesai dibersihkan!
---> Memproses [5/18]: 03.jpg...
✅ 03.jpg selesai dibersihkan!
---> Memproses [6/18]: 17.jpg...
✅ 17.jpg selesai dibersihkan!
---> Memproses [7/18]: 01.jpg...
✅ 01.jpg selesai dibersihkan!
---> Memproses [8/18]: 07.jpg...
✅ 07.jpg selesai dibersihkan!
---> Memproses [9/18]: 18.jpg...
✅ 18.jpg selesai dibersihkan!
---> Memproses [10/18]: 06.jpg...
✅ 06.jpg selesai dibersihkan!
---> Memproses [11/18]: 11.jpg...
✅ 11.jpg selesai dibersihkan!
---> Memproses [12/18]: 15.jpg...
✅ 15.jpg selesai dibersihkan!
---> Memp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 === SEMUA PROSES SELESAI! ===
